# Predição da próxima palavra com uma LSTM

Este notebook apresenta um exemplo de previsão da próxima palavra usando uma rede LSTM (Long Short-Term Memory) com TensorFlow/Keras.

**Ideia:** fornecemos algumas palavras como contexto (por exemplo, `o gato`) e o modelo aprende a prever a palavra seguinte (por exemplo, `dorme`).

> **Importante:** o conjunto de frases é pequeno e criado apenas para demonstração. O modelo pode memorizar padrões e não terá a capacidade de um modelo de linguagem treinado em grandes volumes de texto.

## 1. Importar bibliotecas e definir frases

Para simplificar, a tokenização será feita por palavras e a pontuação será separada previamente de forma básica.

In [1]:
import re
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

frases = [
    "o gato dorme no sofá macio",
    "o gato gosta de leite quente",
    "o gato brinca no jardim ensolarado",
    "o cachorro dorme no quintal",
    "o cachorro gosta de brincar",
    "o cachorro corre no parque",
    "a menina gosta de música",
    "a menina lê um livro",
    "a menina brinca no jardim",
    "o menino lê um livro",
    "o menino gosta de futebol",
    "o menino corre no parque",
    "o sol brilha no céu",
    "a lua aparece no céu",
    "o pássaro canta na árvore",
    "o pássaro voa no céu",
    "a professora lê um livro",
    "a professora gosta de ensinar",
    "o estudante lê um livro",
    "o estudante gosta de aprender",
    "o gato corre no jardim",
    "o cachorro brinca no parque",
    "a menina canta uma música",
    "o menino canta uma música",
    "o café está muito quente",
    "o chá está muito quente",
    "a água está muito fria",
    "o dia está muito bonito",
    "a noite está muito bonita",
    "o gato está muito quieto",

    # Animais
    "o gato bebe água",
    "o gato sobe na árvore",
    "o cachorro protege a casa",
    "o cachorro bebe água",
    "o pássaro constrói um ninho",
    "o peixe nada no rio",
    "a tartaruga anda devagar",
    "o cavalo corre no campo",
    "a borboleta pousa na flor",
    "o coelho come uma cenoura",

    # Pessoas e atividades
    "a menina estuda matemática",
    "o menino aprende português",
    "a professora explica a matéria",
    "o estudante faz a tarefa",
    "o médico trabalha no hospital",
    "a enfermeira cuida dos pacientes",
    "o cozinheiro prepara o jantar",
    "o agricultor planta milho",
    "o músico toca violão",
    "a criança desenha uma casa",

    # Casa e objetos
    "a mãe prepara o almoço",
    "o pai lê o jornal",
    "a família assiste televisão",
    "o bebê dorme no berço",
    "a porta está aberta",
    "a janela está fechada",
    "o relógio marca as horas",
    "o telefone toca muito alto",
    "a lâmpada ilumina o quarto",
    "o computador está ligado",

    # Natureza e clima
    "a chuva molha a rua",
    "o vento balança as árvores",
    "o sol aquece a terra",
    "as nuvens cobrem o céu",
    "o rio atravessa a cidade",
    "as flores crescem no jardim",
    "a grama está muito verde",
    "a neve cobre a montanha",
    "o mar está tranquilo",
    "a tempestade chega à noite",

    # Comida e bebidas
    "o café está quente",
    "o leite está frio",
    "a menina come uma maçã",
    "o menino bebe suco de laranja",
    "a família prepara um bolo",
    "o pão está fresco",
    "a sopa está muito quente",
    "o cozinheiro corta os legumes",
    "o arroz acompanha o feijão",
    "a água mata a sede",

    # Tecnologia e ciência
    "o computador processa os dados",
    "o celular recebe uma mensagem",
    "o programa executa uma tarefa",
    "a rede transmite informações",
    "o sensor mede a temperatura",
    "o robô aprende uma tarefa",
    "a câmera captura imagens",
    "o satélite observa a Terra",
    "a inteligência artificial reconhece padrões",
    "o pesquisador analisa os resultados",
]

print(f"Número de frases: {len(frases)}")
print("Exemplos:")
for frase in frases[:5]:
    print(" •", frase)

Número de frases: 90
Exemplos:
 • o gato dorme no sofá macio
 • o gato gosta de leite quente
 • o gato brinca no jardim ensolarado
 • o cachorro dorme no quintal
 • o cachorro gosta de brincar


## 2. Transformar palavras em números

Uma rede neural não recebe palavras diretamente.

O `Tokenizer` cria um vocabulário e **associa um índice inteiro a cada palavra**.

Em seguida, geramos exemplos de treino com todos os prefixos de cada frase.

Exemplos de treino para a frase `o gato dorme no sofá`:
- `o` → `gato`
- `o gato` → `dorme`
- `o gato dorme` → `no`
- `o gato dorme no` → `sofá`

Todos os exemplos são completados até o mesmo comprimento usando padding.



In [2]:
tokenizer = Tokenizer(oov_token="<desconhecida>")
tokenizer.fit_on_texts(frases)
vocab_size = len(tokenizer.word_index) + 1  # índice 0 reservado para padding

sequencias = []
for frase in frases:
    ids = tokenizer.texts_to_sequences([frase])[0]
    for i in range(1, len(ids)):
        sequencias.append(ids[:i + 1])

max_len = max(len(seq) for seq in sequencias)
sequencias_pad = pad_sequences(sequencias, maxlen=max_len, padding="pre") # completa cada sequência com zeros até o comprimento da maior (max_len = 6: 5 palavras de contexto + 1 alvo).

X = sequencias_pad[:, :-1]  # contexto: todas as palavras menos a última
y = sequencias_pad[:, -1]   # alvo: próxima palavra

print("Tamanho do vocabulário:", vocab_size)
print("Número de exemplos de treino:", len(X))
print("Comprimento máximo do contexto (i.e., sequência de entrada):", X.shape[1])

# Inspecione alguns exemplos em formato legível
indice_para_palavra = {idx: palavra for palavra, idx in tokenizer.word_index.items()}
for x_ex, y_ex in zip(X[:5], y[:5]):
    contexto = [indice_para_palavra.get(int(i), "<PAD>") for i in x_ex if i != 0]
    alvo = indice_para_palavra.get(int(y_ex), "?")
    print(f"{' '.join(contexto):25s} -> {alvo}")

Tamanho do vocabulário: 190
Número de exemplos de treino: 347
Comprimento máximo do contexto (i.e., sequência de entrada): 5
o                         -> gato
o gato                    -> dorme
o gato dorme              -> no
o gato dorme no           -> sofá
o gato dorme no sofá      -> macio


### Palavra → índice

In [3]:
tokenizer.word_index

{'<desconhecida>': 1,
 'o': 2,
 'a': 3,
 'no': 4,
 'está': 5,
 'muito': 6,
 'uma': 7,
 'gato': 8,
 'de': 9,
 'gosta': 10,
 'cachorro': 11,
 'menina': 12,
 'um': 13,
 'menino': 14,
 'quente': 15,
 'lê': 16,
 'jardim': 17,
 'corre': 18,
 'livro': 19,
 'céu': 20,
 'água': 21,
 'as': 22,
 'dorme': 23,
 'brinca': 24,
 'parque': 25,
 'música': 26,
 'pássaro': 27,
 'canta': 28,
 'na': 29,
 'professora': 30,
 'estudante': 31,
 'bebe': 32,
 'tarefa': 33,
 'prepara': 34,
 'os': 35,
 'leite': 36,
 'sol': 37,
 'árvore': 38,
 'café': 39,
 'noite': 40,
 'casa': 41,
 'rio': 42,
 'come': 43,
 'aprende': 44,
 'cozinheiro': 45,
 'toca': 46,
 'família': 47,
 'computador': 48,
 'terra': 49,
 'sofá': 50,
 'macio': 51,
 'ensolarado': 52,
 'quintal': 53,
 'brincar': 54,
 'futebol': 55,
 'brilha': 56,
 'lua': 57,
 'aparece': 58,
 'voa': 59,
 'ensinar': 60,
 'aprender': 61,
 'chá': 62,
 'fria': 63,
 'dia': 64,
 'bonito': 65,
 'bonita': 66,
 'quieto': 67,
 'sobe': 68,
 'protege': 69,
 'constrói': 70,
 'ninho': 

## 3. Criar e treinar a LSTM

A camada `Embedding` aprende uma **representação vetorial para cada palavra**: vetores de 32 dimensões para cada palavra.

A LSTM processa a sequência de palavras na ordem em que aparece e a camada `Dense` produz uma pontuação para cada palavra do vocabulário.

Com `softmax`, essas pontuações podem ser interpretadas como probabilidades.

**Contexto (entrada):** as palavras anteriores são a entrada do modelo.

**Alvo (rótulo):** a palavra seguinte é a resposta esperada durante o treino.

In [4]:
model = Sequential(
    [
        # Informa às camadas seguintes que os zeros são apenas enchimento. A LSTM então ignora essas posições e não trata os zeros como palavras.
        Embedding(input_dim=vocab_size, output_dim=32, mask_zero=True),
        # Cria uma camada LSTM com 64 unidades. O número 64 é a dimensão do estado interno da camada: a quantidade de valores que ela usa para “lembrar” o que já leu da sequência.
        LSTM(64),
        # A camada densa calcula as pontuações para cada palavra do vocabulário que, na sequência, são normalizadas pela função softmax.
        Dense(vocab_size, activation="softmax")
    ]
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [5]:
history = model.fit(
    X, y,
    epochs=250,
    batch_size=16,
    verbose=0
)

print("Treinamento concluído.")
print(f"Acurácia no próprio conjunto de treino: {history.history['accuracy'][-1]:.3f}")

Treinamento concluído.
Acurácia no próprio conjunto de treino: 0.657


**OBS**.: Essa acurácia é calculada nos dados usados para treinar. Ela não mede generalização.

## 4. Predizer a próxima palavra

A função abaixo recebe um contexto de palavras, transforma-o em números, consulta o modelo e retorna as palavras com maior probabilidade.

In [6]:
def prever_proxima_palavra(contexto, top_k=5):
    ids = tokenizer.texts_to_sequences([contexto])[0]
    if not ids:
        raise ValueError("Digite ao menos uma palavra conhecida pelo tokenizador.")

    entrada = pad_sequences([ids], maxlen=max_len - 1, padding="pre")
    probabilidades = model.predict(entrada, verbose=0)[0]

    # Não sugerir o índice 0 (padding)
    indices = np.argsort(probabilidades[1:])[::-1][:top_k] + 1
    return [
        (indice_para_palavra.get(int(i), "?"), float(probabilidades[i]))
        for i in indices
    ]

contextos_teste = [
    "o gato",
    "o cachorro",
    "a menina",
    "o menino lê",
    "o pássaro"
]

for contexto in contextos_teste:
    print(f"Contexto: {contexto!r}")
    for palavra, prob in prever_proxima_palavra(contexto, top_k=3):
        print(f"   {palavra:15s} probabilidade = {prob:.3f}")
    print()

Contexto: 'o gato'
   gosta           probabilidade = 0.150
   brinca          probabilidade = 0.141
   dorme           probabilidade = 0.140

Contexto: 'o cachorro'
   corre           probabilidade = 0.176
   dorme           probabilidade = 0.166
   brinca          probabilidade = 0.164

Contexto: 'a menina'
   come            probabilidade = 0.167
   estuda          probabilidade = 0.166
   lê              probabilidade = 0.165

Contexto: 'o menino lê'
   um              probabilidade = 0.990
   suco            probabilidade = 0.002
   água            probabilidade = 0.002

Contexto: 'o pássaro'
   constrói        probabilidade = 0.331
   canta           probabilidade = 0.312
   voa             probabilidade = 0.312



## 5. Experimente seus próprios contextos

Você pode editar o texto na célula seguinte e executá-la.

Como o vocabulário é pequeno, palavras desconhecidas podem ser ignoradas pelo tokenizador.

A previsão é apenas uma demonstração do mecanismo, não uma resposta garantidamente correta.

In [7]:
contexto = "o gato dorme"
print("Contexto:", contexto)
print("Próximas palavras mais prováveis:")
for palavra, prob in prever_proxima_palavra(contexto, top_k=5):
    print(f"  {palavra:15s} {prob:.3f}")

Contexto: o gato dorme
Próximas palavras mais prováveis:
  no              0.996
  na              0.003
  a               0.000
  uma             0.000
  muito           0.000


In [8]:
contexto = "o gato dorme no"
print("Contexto:", contexto)
print("Próximas palavras mais prováveis:")
for palavra, prob in prever_proxima_palavra(contexto, top_k=5):
    print(f"  {palavra:15s} {prob:.3f}")

Contexto: o gato dorme no
Próximas palavras mais prováveis:
  sofá            0.947
  quintal         0.015
  jardim          0.011
  berço           0.004
  tarefa          0.004


In [9]:
contexto = "o gato dorme no sofá"
print("Contexto:", contexto)
print("Próximas palavras mais prováveis:")
for palavra, prob in prever_proxima_palavra(contexto, top_k=5):
    print(f"  {palavra:15s} {prob:.3f}")

Contexto: o gato dorme no sofá
Próximas palavras mais prováveis:
  macio           0.990
  ensolarado      0.002
  sofá            0.001
  tarefa          0.001
  laranja         0.001
